In [ ]:
import torch
import matplotlib.pyplot as plt

from bhtrace.geometry import Photon, Observer
import bhtrace.geometry.spacetime as st
from bhtrace.tracing import PTracer

import bhtrace.medium as bhM
import bhtrace.graphics as bhg

from bhtrace.grrt import GRRT, Blackbody

## 1. Running ray-tracing

In [ ]:
N = 128  # Image resolution
D = 32
KERR_A = 0.95
OBS_R = 20.0
OBS_INCLINATION = torch.pi / 180 * 84
TRACE_T = 36
TRACE_NSTEPS = 256
TRACE_R_MAX = 25
TRACE_EPS = 1e-4

base = st.KerrBL(a=KERR_A)
photon = Photon(spacetime=base)
obs = Observer(
    spacetime=base,
    r=OBS_R,
    inclination=OBS_INCLINATION,
    u=torch.Tensor([1, 0, 0, 0]),
)
obs.generate_net(net_shape="square", net_rng=(N, N), net_size=(D, D))
X0, P0 = obs.setup_ic(photon)

tracer = PTracer(ode_method="VCABM4")

dev = "cuda" if torch.cuda.is_available() else 'cpu'
print(f"Using device: {dev}")

tracer.__const_dx__ = True
traj = tracer.forward(
    photon, X0, P0, T=TRACE_T, nsteps=TRACE_NSTEPS, r_max=TRACE_R_MAX, eps=TRACE_EPS, device=dev
)

## 2. Running GRRT

In [ ]:
medium = bhM.KeplerianDisk(base, 1e8, mass_dot=1e-1, mu=1.0,)

frequences = torch.logspace(1, 12, 32)
model = Blackbody(frequences)

grrt = GRRT(medium=medium)
grrt.set_models(model)

shoot = grrt.compute(traj, history=True, dtype=torch.float32, device=dev)

## 3. Plotting results

#### 3.1 Spectrum

In [ ]:
flux_spectral = model.intensities.x.cpu()
spectrum = flux_spectral.sum(dim=0)
print(spectrum)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

log_nu = frequences.log10().numpy() 
log_I = spectrum.log10().numpy()

excepted = log_I[0] + 4/3 * (log_nu - log_nu[0])

ax.plot(log_nu, log_I)
ax.plot(log_nu, excepted, linestyle='--')
ax.set_xlabel('log10_nu')
ax.set_ylabel('log10_flux')
ax.grid('on')

plt.show()
plt.show()

### 3.2 Spectrally aggregated image

In [ ]:
agg_image = flux_spectral.sum(dim=-1)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
image = agg_image.cpu().view(N, N).transpose(-1, -2)
imA = ax.imshow(image, cmap="afmhot")

fig.colorbar(imA, ax=ax)
plt.show()

### 3.3 Spectral images

In [ ]:
spectral_images = model.intensities.x.clip(min=0.0).log1p()

vmin = spectral_images.min()
vmax = spectral_images.max()

threshold = 0.2*vmax

for i in range(len(frequences)):

    if spectrum[i].log1p() > threshold:

        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
        image = spectral_images[..., i].cpu().view(N, N).transpose(-1, -2)
        imA = ax.imshow(image, cmap="afmhot", vmin=vmin, vmax=vmax)
        ax.set_title(f"nu = {frequences[i]:.3e} Hz")
        
        fig.colorbar(imA, ax=ax)
        plt.show()